# Data Cleaning Notebook

This notebook prepares the raw salary dataset for model training. The raw Kaggle CSV is not modified. The cleaned output removes invalid rows, trims extreme salary outliers, groups rare job titles into `Other`, and writes a model-ready CSV.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)

raw_path = Path("../data/raw/ds_salaries.csv")
df = pd.read_csv(raw_path)

print(f"Raw shape: {df.shape}")
print(df.head())


Raw shape: (607, 12)
   Unnamed: 0  work_year experience_level employment_type  \
0           0       2020               MI              FT   
1           1       2020               SE              FT   
2           2       2020               SE              FT   
3           3       2020               MI              FT   
4           4       2020               SE              FT   

                    job_title  salary salary_currency  salary_in_usd  \
0              Data Scientist   70000             EUR          79833   
1  Machine Learning Scientist  260000             USD         260000   
2           Big Data Engineer   85000             GBP         109024   
3        Product Data Analyst   20000             USD          20000   
4   Machine Learning Engineer  150000             USD         150000   

  employee_residence  remote_ratio company_location company_size  
0                 DE             0               DE            L  
1                 JP             0           

## Standardize Columns

Normalize column names so the rest of the pipeline can reference stable field names.


In [2]:
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
print(df.columns.tolist())


['unnamed:_0', 'work_year', 'experience_level', 'employment_type', 'job_title', 'salary', 'salary_currency', 'salary_in_usd', 'employee_residence', 'remote_ratio', 'company_location', 'company_size']


## Missing Values and Duplicates

The dataset should not contain missing values for this project. Rows with missing fields are removed, then exact duplicate rows are dropped.


In [3]:
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2),
}).sort_values("missing_count", ascending=False)
print(missing_summary)

before = len(df)
df = df.dropna().drop_duplicates()
print(f"Rows removed for missing values or duplicates: {before - len(df)}")
print(f"Shape after missing/duplicate cleanup: {df.shape}")


                    missing_count  missing_percent
unnamed:_0                      0              0.0
work_year                       0              0.0
experience_level                0              0.0
employment_type                 0              0.0
job_title                       0              0.0
salary                          0              0.0
salary_currency                 0              0.0
salary_in_usd                   0              0.0
employee_residence              0              0.0
remote_ratio                    0              0.0
company_location                0              0.0
company_size                    0              0.0
Rows removed for missing values or duplicates: 0
Shape after missing/duplicate cleanup: (607, 12)


## Clean Text and Encoded Categories

Trim text columns and replace compact dataset codes with readable labels used by the API and dashboard.


In [4]:
categorical_columns = [
    "experience_level",
    "employment_type",
    "job_title",
    "employee_residence",
    "company_location",
    "company_size",
]

for col in categorical_columns:
    df[col] = df[col].astype(str).str.strip()

experience_map = {
    "EN": "Entry-level",
    "MI": "Mid-level",
    "SE": "Senior-level",
    "EX": "Executive-level",
}

employment_map = {
    "FT": "Full-time",
    "PT": "Part-time",
    "CT": "Contract",
    "FL": "Freelance",
}

company_size_map = {"S": "Small", "M": "Medium", "L": "Large"}

df["experience_level"] = df["experience_level"].replace(experience_map)
df["employment_type"] = df["employment_type"].replace(employment_map)
df["company_size"] = df["company_size"].replace(company_size_map)

for col in ["experience_level", "employment_type", "company_size"]:
    print(f"\n{col}")
    print(df[col].value_counts())



experience_level
experience_level
Senior-level       280
Mid-level          213
Entry-level         88
Executive-level     26
Name: count, dtype: int64

employment_type
employment_type
Full-time    588
Part-time     10
Contract       5
Freelance      4
Name: count, dtype: int64

company_size
company_size
Medium    326
Large     198
Small      83
Name: count, dtype: int64


## Numeric Fields

Coerce the numeric fields and remove invalid salary rows. Salary values must be positive because `salary_in_usd` is the model target.


In [5]:
numeric_columns = ["work_year", "salary", "salary_in_usd", "remote_ratio"]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

before = len(df)
df = df.dropna(subset=numeric_columns)
df = df[(df["salary"] > 0) & (df["salary_in_usd"] > 0)]

df["work_year"] = df["work_year"].astype(int)
df["remote_ratio"] = df["remote_ratio"].astype(int)

print(f"Rows removed for invalid numeric values: {before - len(df)}")
print(df[numeric_columns].describe())


Rows removed for invalid numeric values: 0
         work_year        salary  salary_in_usd  remote_ratio
count   607.000000  6.070000e+02     607.000000     607.00000
mean   2021.405272  3.240001e+05  112297.869852      70.92257
std       0.692133  1.544357e+06   70957.259411      40.70913
min    2020.000000  4.000000e+03    2859.000000       0.00000
25%    2021.000000  7.000000e+04   62726.000000      50.00000
50%    2022.000000  1.150000e+05  101570.000000     100.00000
75%    2022.000000  1.650000e+05  150000.000000     100.00000
max    2022.000000  3.040000e+07  600000.000000     100.00000


## Salary Outlier Check

Extreme salary values can dominate tree-based regression models on a small dataset. The cap below removes only salaries above the 99th percentile of `salary_in_usd`. This keeps the method simple, explainable, and based only on the target distribution.


In [6]:
salary_quantiles = df["salary_in_usd"].quantile([0.50, 0.75, 0.90, 0.95, 0.99, 1.00])
print("Salary quantiles before outlier removal:")
print(salary_quantiles)

salary_99th_percentile = df["salary_in_usd"].quantile(0.99)
outlier_rows = df[df["salary_in_usd"] > salary_99th_percentile]
print(f"99th percentile threshold: ${salary_99th_percentile:,.2f}")
print(f"Extreme salary rows to remove: {len(outlier_rows)}")
if not outlier_rows.empty:
    print(outlier_rows[["work_year", "job_title", "experience_level", "salary_in_usd"]].sort_values("salary_in_usd", ascending=False))

df = df[df["salary_in_usd"] <= salary_99th_percentile].copy()
print(f"Shape after outlier removal: {df.shape}")


Salary quantiles before outlier removal:
0.50    101570.0
0.75    150000.0
0.90    200000.0
0.95    220110.0
0.99    403500.0
1.00    600000.0
Name: salary_in_usd, dtype: float64
99th percentile threshold: $403,500.00
Extreme salary rows to remove: 7
     work_year                           job_title experience_level  \
252       2021             Principal Data Engineer  Executive-level   
97        2021              Financial Data Analyst        Mid-level   
33        2020                  Research Scientist        Mid-level   
157       2021  Applied Machine Learning Scientist        Mid-level   
225       2021            Principal Data Scientist  Executive-level   
63        2020                      Data Scientist     Senior-level   
523       2022                 Data Analytics Lead     Senior-level   

     salary_in_usd  
252         600000  
97          450000  
33          450000  
157         423000  
225         416000  
63          412000  
523         405000  
Shape after 

## Group Rare Job Titles

Very rare job titles create sparse one-hot columns and make the model sensitive to individual records. Titles with fewer than five records are grouped into `Other`.


In [7]:
rare_job_threshold = 5
job_counts = df["job_title"].value_counts()
rare_job_titles = job_counts[job_counts < rare_job_threshold].index

print(f"Unique job titles before grouping: {job_counts.size}")
print(f"Rare job titles grouped into Other: {len(rare_job_titles)}")

df["job_title"] = np.where(df["job_title"].isin(rare_job_titles), "Other", df["job_title"])

print(f"Unique job titles after grouping: {df['job_title'].nunique()}")
print(df["job_title"].value_counts().head(20))


Unique job titles before grouping: 49
Rare job titles grouped into Other: 27
Unique job titles after grouping: 23
job_title
Data Scientist                142
Data Engineer                 132
Data Analyst                   97
Other                          56
Machine Learning Engineer      41
Research Scientist             15
Data Science Manager           12
Data Architect                 11
Machine Learning Scientist      8
Big Data Engineer               8
Data Science Consultant         7
Director of Data Science        7
AI Scientist                    7
Data Analytics Manager          7
Lead Data Engineer              6
BI Data Analyst                 6
ML Engineer                     6
Computer Vision Engineer        6
Principal Data Scientist        6
Business Data Analyst           5
Name: count, dtype: int64


## Work Setting Label

Keep `remote_ratio` as the model feature and add `work_setting` for readability in analysis and dashboard summaries.


In [8]:
remote_map = {0: "On-site", 50: "Hybrid", 100: "Remote"}
df = df[df["remote_ratio"].isin(remote_map)].copy()
df["work_setting"] = df["remote_ratio"].map(remote_map)

print(df[["remote_ratio", "work_setting"]].drop_duplicates().sort_values("remote_ratio"))


   remote_ratio work_setting
0             0      On-site
2            50       Hybrid
5           100       Remote


## Build and Save the Model-Ready Dataset

Select the fields used by training and application analysis, run final validation checks, and save the processed CSV.


In [9]:
model_columns = [
    "work_year",
    "experience_level",
    "employment_type",
    "job_title",
    "employee_residence",
    "remote_ratio",
    "work_setting",
    "company_location",
    "company_size",
    "salary_in_usd",
]

model_df = df[model_columns].copy()
model_df = model_df.drop_duplicates().reset_index(drop=True)

print(f"Final model dataset shape: {model_df.shape}")
print("Missing values in final dataset:")
print(model_df.isna().sum())
print("Final salary range:")
print(model_df["salary_in_usd"].describe())

processed_path = Path("../data/processed/cleaned_salaries.csv")
processed_path.parent.mkdir(parents=True, exist_ok=True)
model_df.to_csv(processed_path, index=False)

print(f"Cleaned dataset saved to: {processed_path}")


Final model dataset shape: (558, 10)
Missing values in final dataset:
work_year             0
experience_level      0
employment_type       0
job_title             0
employee_residence    0
remote_ratio          0
work_setting          0
company_location      0
company_size          0
salary_in_usd         0
dtype: int64
Final salary range:
count       558.000000
mean     106342.014337
std       61376.021956
min        2859.000000
25%       60000.000000
50%      100000.000000
75%      147600.000000
max      380000.000000
Name: salary_in_usd, dtype: float64
Cleaned dataset saved to: ..\data\processed\cleaned_salaries.csv
